In [3]:
from src.models.features import build_training_features
df = build_training_features()
df.to_csv("data/training_features.csv", index=False)

Built 15,872 training samples with 34 features


In [4]:
import pandas as pd
df = pd.read_csv("data/training_features.csv")

In [5]:
df.shape

(15872, 34)

In [6]:
df.tail(10)

,date,home_team,away_team,tournament,neutral,result,home_ranking,away_ranking,ranking_diff,home_elo,...,away_form_gf,away_form_ga,away_form_gd,h2h_matches,h2h_home_wins,h2h_away_wins,h2h_draws,home_days_rest,away_days_rest,is_world_cup
15862,2026-06-21,Spain,Saudi Arabia,FIFA World Cup,1,HOME,2.0,57.0,55.0,2165.0,...,6,5,1,3,3,0,0,6,6,1
15863,2026-06-21,Uruguay,Cape Verde,FIFA World Cup,1,DRAW,17.0,70.0,53.0,1892.0,...,6,8,-2,0,0,0,0,6,6,1
15864,2026-06-21,Ecuador,Curaçao,FIFA World Cup,1,DRAW,24.0,81.0,57.0,1933.0,...,7,18,-11,0,0,0,0,7,7,1
15865,2026-06-22,Argentina,Austria,FIFA World Cup,1,HOME,3.0,23.0,20.0,2113.0,...,13,3,10,2,1,0,1,5,5,1
15866,2026-06-22,New Zealand,Egypt,FIFA World Cup,1,AWAY,95.0,29.0,-66.0,1585.0,...,7,3,4,3,0,2,1,6,7,1
15867,2026-06-22,France,Iraq,FIFA World Cup,1,HOME,1.0,61.0,60.0,2081.0,...,5,8,-3,0,0,0,0,6,6,1
15868,2026-06-23,Norway,Senegal,FIFA World Cup,1,HOME,44.0,14.0,-30.0,1912.0,...,8,7,1,1,0,1,0,7,7,1
15869,2026-06-23,Portugal,Uzbekistan,FIFA World Cup,1,HOME,5.0,62.0,57.0,1984.0,...,3,10,-7,0,0,0,0,6,5,1
15870,2026-06-23,Jordan,Algeria,FIFA World Cup,1,AWAY,68.0,36.0,-32.0,1690.0,...,5,6,-1,2,0,1,1,6,6,1
15871,2026-06-23,England,Ghana,FIFA World Cup,1,DRAW,4.0,65.0,61.0,2020.0,...,4,10,-6,1,0,0,1,6,6,1


In [7]:
df.head()

,date,home_team,away_team,tournament,neutral,result,home_ranking,away_ranking,ranking_diff,home_elo,...,away_form_gf,away_form_ga,away_form_gd,h2h_matches,h2h_home_wins,h2h_away_wins,h2h_draws,home_days_rest,away_days_rest,is_world_cup
0,2010-01-02,Iran,North Korea,Friendly,1,HOME,21.0,NaN,NaN,1760.0,...,11,4,7,10,8,0,2,3,3,0
1,2010-01-02,Qatar,Mali,Friendly,0,DRAW,35.0,NaN,65.0,1425.0,...,6,5,1,0,0,0,0,3,3,0
2,2010-01-02,Syria,Zimbabwe,Friendly,1,HOME,NaN,NaN,0.0,NaN,...,7,7,0,0,0,0,0,3,4,0
3,2010-01-02,Yemen,Tajikistan,Friendly,0,AWAY,NaN,NaN,0.0,NaN,...,7,11,-4,1,1,0,0,3,3,0
4,2010-01-03,Angola,Gambia,Friendly,1,DRAW,NaN,NaN,NaN,NaN,...,5,2,3,0,0,0,0,4,180,0


In [8]:
df.columns

Index(['date', 'home_team', 'away_team', 'tournament', 'neutral', 'result',
       'home_ranking', 'away_ranking', 'ranking_diff', 'home_elo', 'away_elo',
       'elo_diff', 'home_squad_age', 'away_squad_age', 'home_market_value',
       'away_market_value', 'market_value_ratio', 'home_form_win_rate',
       'home_form_pts_rate', 'home_form_gf', 'home_form_ga', 'home_form_gd',
       'away_form_win_rate', 'away_form_pts_rate', 'away_form_gf',
       'away_form_ga', 'away_form_gd', 'h2h_matches', 'h2h_home_wins',
       'h2h_away_wins', 'h2h_draws', 'home_days_rest', 'away_days_rest',
       'is_world_cup'],
      dtype='object')

In [9]:
print(len(df.columns))

34


In [1]:
from src.data_pipeline.db import get_db, queries
import pandas as pd

with get_db() as conn:
    # New match results from the API
    print(queries.finished_matches(conn))

    # Odds that were just fetched
    print(queries.latest_odds(conn))

    # Team form stats
    print(pd.read_sql_query("SELECT * FROM team_stats LIMIT 10", conn))

    # Pipeline run log — shows every job and whether it succeeded
    print(pd.read_sql_query("SELECT * FROM pipeline_log ORDER BY finished_at DESC LIMIT 10", conn))

          date        stage group_name     home_team               away_team  \
0   2026-06-11  Group Stage          A        Mexico            South Africa   
1   2026-06-12  Group Stage          B        Canada  Bosnia and Herzegovina   
2   2026-06-12  Group Stage    GROUP A   South Korea          Czech Republic   
3   2026-06-13  Group Stage          B         Qatar             Switzerland   
4   2026-06-13  Group Stage          C        Brazil                 Morocco   
..         ...          ...        ...           ...                     ...   
68  2026-06-27  Group Stage    GROUP G   New Zealand                 Belgium   
69  2026-06-27  Group Stage    GROUP G         Egypt                    Iran   
70  2026-06-28  Group Stage    GROUP J        Jordan               Argentina   
71  2026-06-28  Group Stage    GROUP J       Algeria                 Austria   
72  2026-06-28      Last 32       None  South Africa                  Canada   

    home_score  away_score            v

## Odds — qu'est-ce qu'on a ?

On a la *plomberie* (scraper + table SQLite) mais **aucune donnée de cotes** pour l'instant.
Les cellules ci-dessous montrent l'état réel et comment des cotes se transforment en probabilités.


### 1. État de la table `odds` dans la base

In [3]:
import sqlite3
import pandas as pd
from src.data_pipeline.db import DB_PATH

print("Base:", DB_PATH)
conn = None
try:
    conn = sqlite3.connect(DB_PATH)
    n = conn.execute("SELECT COUNT(*) FROM odds").fetchone()[0]
    print(f"Lignes dans la table 'odds': {n}")
    if n:
        display(pd.read_sql_query(
            "SELECT     home_team,    away_team,    AVG(home_win) AS avg_home_win,    AVG(draw) AS avg_draw,    AVG(away_win) AS avg_away_win,    COUNT(*) AS nb_scrapes,    MAX(scraped_at) AS last_scrape FROM odds GROUP BY home_team, away_team ORDER BY last_scrape DESC;", conn))
    else:
        print("\u2192 La table existe mais est VIDE : aucune cote n'a encore \u00e9t\u00e9 r\u00e9cup\u00e9r\u00e9e.")
except sqlite3.DatabaseError as e:
    print(f"\u26a0 Impossible de lire la base : {e}")
    print("\u2192 SQLite corrompue/absente. \u00c0 r\u00e9g\u00e9n\u00e9rer : python -m src.scripts.ingest_historical")
finally:
    if conn is not None:
        conn.close()

Base: C:\Users\Amélie\Documents\BeCode\Projects\world_cup_prediction\data\football.db
Lignes dans la table 'odds': 6026


,home_team,away_team,avg_home_win,avg_draw,avg_away_win,nb_scrapes,last_scrape
0,Argentina,Cape Verde,1.155600,8.042000,19.180400,25,2026-06-29 12:00:44
1,Australia,Egypt,3.344800,2.935600,2.475600,25,2026-06-29 12:00:44
2,Belgium,Senegal,2.215200,3.264800,3.499600,25,2026-06-29 12:00:44
3,Brazil,Japan,1.729600,3.734400,5.219600,25,2026-06-29 12:00:44
4,Colombia,Ghana,1.524000,3.972000,7.380400,25,2026-06-29 12:00:44
5,Côte d'Ivoire,Norway,3.698000,3.588800,2.023600,25,2026-06-29 12:00:44
6,England,Congo DR,1.270870,5.642609,13.319130,23,2026-06-29 12:00:44
7,France,Sweden,1.269200,6.223600,11.073600,25,2026-06-29 12:00:44
8,Germany,Paraguay,1.363200,5.072400,9.224000,25,2026-06-29 12:00:44
9,Mexico,Ecuador,2.212800,2.952400,3.984400,25,2026-06-29 12:00:44


### 2. Le scraper d'odds et la clé API

In [ ]:
from src.utils.config import get_config
from src.data_pipeline.scraper import OddsScraper

cfg = get_config()
has_key = bool(cfg.get("odds_api_key"))
print("ODDS_API_KEY configurée :", has_key)

# fetch_odds() peut échouer (pas de clé, réseau, ou cotes WC pas encore publiées).
rows = []
try:
    rows = OddsScraper().fetch_odds(api_key=cfg.get("odds_api_key"))
    print("Cotes récupérées via l'API :", len(rows))
except Exception as e:
    print("Appel API impossible ici :", type(e).__name__, "-", e)

if rows:
    display(pd.DataFrame(rows).head())

### 3. Convertir une cote en probabilité

Cote décimale → proba implicite = `1 / cote`. La somme dépasse 100% (la marge du
bookmaker, *overround*) : on divise par le total pour « dé-marger » et retomber à 100%.


In [ ]:
def implied_probabilities(decimals: dict) -> dict:
    """Cotes d\u00e9cimales -> probabilit\u00e9s implicites d\u00e9-marg\u00e9es (somme = 1)."""
    raw = {k: 1.0 / v for k, v in decimals.items()}
    total = sum(raw.values())          # > 1 : c'est la marge du bookmaker
    return {k: p / total for k, p in raw.items()}

# Exemple 1X2 d'un match (cotes fictives)
match_odds = {"home_win": 1.80, "draw": 3.50, "away_win": 4.50}
overround = sum(1 / v for v in match_odds.values()) * 100 - 100
print(f"Marge bookmaker (overround) : {overround:.1f}%")

probs = implied_probabilities(match_odds)
pd.DataFrame({
    "cote": match_odds,
    "proba_implicite_%": {k: round(v * 100, 1) for k, v in probs.items()},
})

### 4. Cotes « outright » = ce qu'il faut pour le dashboard

Le dashboard classe les équipes par probabilité de **gagner la Coupe**. Voici à quoi
ressembleraient ces données (cotes fictives) une fois converties en probabilité de marché —
la colonne qu'on pourrait afficher à côté de la prédiction du modèle.


In [ ]:
# Cotes outright (vainqueur du tournoi) — FICTIVES, juste pour visualiser le format
outright = {
    "France": 5.0, "Brazil": 6.0, "England": 7.5, "Spain": 8.0,
    "Argentina": 9.0, "Germany": 11.0, "Portugal": 13.0, "Netherlands": 15.0,
}
probs = implied_probabilities(outright)
df_market = (
    pd.DataFrame({"decimal_odds": outright})
      .assign(market_prob_pct=lambda d: [round(probs[t] * 100, 1) for t in d.index])
      .sort_values("market_prob_pct", ascending=False)
)
df_market

### 5. (Optionnel) Générer un CSV à remplir

Si tu n'as pas de clé API, tu peux saisir les cotes outright à la main. Cette cellule
écrit un gabarit `data/raw/odds_outright_TEMPLATE.csv` (équipes du Mondial, colonne `decimal_odds` vide à compléter).


In [ ]:
from pathlib import Path
import pandas as pd

teams = sorted(pd.read_csv("data/raw/wc_2026_teams.csv")["team"])
template = pd.DataFrame({"team": teams, "decimal_odds": ""})
out = Path("data/raw/odds_outright_TEMPLATE.csv")
template.to_csv(out, index=False)
print(f"Gabarit \u00e9crit : {out}  ({len(teams)} \u00e9quipes)")
template.head()